In [1]:
import numpy as np
import pandas as pd
import os
import sys
sys.path.append("C:/Users/wg984/Wolfgang/repos/Bedmaster-ICU-tools/code/")
from bmresearch_tools import BMR_load, get_metadata
from datetime import datetime, timedelta

In [2]:
bmr_studyid_dir = '//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/data_analysis/BMR_studyID/'
airgo_dir = '//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/data_analysis/AirGo/airgo_research/'
# study table including cam dates:
tablefile = 'M:/Projects/ICU_SLEEP_STUDY/data/data_analysis/Study/ICUSleep_DataTable_ICUonly.csv'
table= pd.read_csv(tablefile)
table.StudyID = table.StudyID.apply(lambda x: str(x).zfill(3))

In [3]:
# get bedmaster data:
def get_bm_data():
    bmr_file = os.path.join(bmr_studyid_dir,'BMR_'+ study_id +'.h5')
    bmr_exists = os.path.exists(bmr_file)
    
    bm_available_day = []
    bm_available_night = []

    if bmr_exists:
        bm_meta = get_metadata(bmr_file)
        signals = [x for x in ['HR','SPO2%'] if x in bm_meta]
        bm_data = BMR_load(bmr_file, signals = signals)
        bm_available_hours = []
        for signal in signals:
            bm_available_hours = list(set(bm_available_hours).union(set(pd.unique(bm_data[signal].datetime.apply(lambda x: str(x.date()) + ' ' + str(x.hour).zfill(2))))))
        bm_available_hours.sort()
        hours_dt = pd.to_datetime(bm_available_hours, infer_datetime_format=1)
    
        bm_available_night = [str((x - timedelta(hours=8)).date()) for x in hours_dt if x.hour in [20,21,22,23,0,1,2,3,4,5,6,7]]
        nights, counts = np.unique(bm_available_night, return_counts=True)
    #     print(nights)
    #     print(counts)
        bm_available_night = []
        for iN, night in enumerate(nights):
            if counts[iN] >= 5: bm_available_night.append(night) 
    #     print(bm_available_night)

        bm_available_day = [str(x.date()) for x in hours_dt if x.hour in [8,9,10,11,12,13,14,15,16,17,18,19]]
        days, counts = np.unique(bm_available_day, return_counts=True)
    #     print(days)
    #     print(counts)
        bm_available_day = []
        for iN, day in enumerate(days):
            if counts[iN] >= 5: bm_available_day.append(day) 
    #     print(bm_available_day)

    return bm_available_day, bm_available_night


    
def get_airgo_data():
    
    airgo_file = os.path.join(airgo_dir, 'airgo_' + study_id + '.csv')
    airgo_exists = os.path.exists(airgo_file)

    airgo_available_night = []
    airgo_available_day = []
    
    if airgo_exists: 
        airgo_data = pd.read_csv(airgo_file)
        airgo_data.DateTime = pd.to_datetime(airgo_data.DateTime, infer_datetime_format=1)

        airgo_available_hours = pd.unique(airgo_data.DateTime.apply(lambda x: str(x.date()) + ' ' + str(x.hour).zfill(2)))
        hours_dt = pd.to_datetime(airgo_available_hours, infer_datetime_format=1)

        airgo_available_night = [str((x - timedelta(hours=8)).date()) for x in hours_dt if x.hour in [20,21,22,23,0,1,2,3,4,5,6,7]]
        nights, counts = np.unique(airgo_available_night, return_counts=True)
#         print(nights)
#         print(counts)
        
        for iN, night in enumerate(nights):
            if counts[iN] >= 5: airgo_available_night.append(night) 
#         print(airgo_available_night)
        airgo_available_day = [str(x.date()) for x in hours_dt if x.hour in [8,9,10,11,12,13,14,15,16,17,18,19]]
        days, counts = np.unique(airgo_available_day, return_counts=True)
#         print(days)
#         print(counts)

        for iN, day in enumerate(days):
            if counts[iN] >= 5: airgo_available_day.append(day) 
#         print(airgo_available_day)

    return airgo_available_day, airgo_available_night


# table
def get_icu_data():
    
    table_studyid = table[table.StudyID == study_id]
    transferin = pd.to_datetime(pd.unique(table_studyid.ADT_TransferIn), infer_datetime_format=1).dropna()
    transferout = pd.to_datetime(pd.unique(table_studyid.ADT_TransferOut), infer_datetime_format=1).dropna()
    cam_dates = pd.to_datetime(table_studyid.CAM_Date.values, infer_datetime_format=1)
    admission = pd.to_datetime(table_studyid.ADT_Admission, infer_datetime_format=1)
    discharge = pd.to_datetime(table_studyid.ADT_Discharge, infer_datetime_format=1)
    nights_in_icu = []
#     print(transferin)
#     print(transferout)
    for iStay in range(len(transferin)):
        first_night = transferin[iStay].date()
        last_day = transferout[iStay].date()
        delta = last_day - first_night
        for i in range(delta.days):
            nights_in_icu.append(str(first_night + timedelta(days=i)))

    cam_icu_morning = [str(x.date()) for x in cam_dates if x.hour <= 12]
    cam_icu_evening = [str(x.date()) for x in cam_dates if x.hour > 12]

    
    return nights_in_icu, cam_icu_morning, cam_icu_evening


def report_for_id():
    
    all_dates = set(nights_in_icu) | set(cam_icu_morning) | set(cam_icu_evening) | set(airgo_available_day) | set(airgo_available_night) | set(bm_available_day) | set(bm_available_night)
    all_dates = list(all_dates)
    all_dates.sort()
    report_id = pd.DataFrame(columns=['study_id', 'date', 'day_night','cam_icu','airgo','bedmaster'])
    all_dates2 = all_dates*2
    all_dates2.sort()
    report_id['date'] = all_dates2
    report_id['study_id'] = study_id
    report_id['day_night'] = ['day','night']*len(all_dates)
    report_id[['cam_icu','airgo','bedmaster']] = False
    for date in all_dates:
        if date in cam_icu_morning: report_id.loc[(report_id.date==date).values & (report_id.day_night=='day').values,'cam_icu'] = True
        if date in cam_icu_evening: report_id.loc[(report_id.date==date).values & (report_id.day_night=='night').values,'cam_icu'] = True
        if date in airgo_available_day: report_id.loc[(report_id.date==date).values & (report_id.day_night=='day').values,'airgo'] = True
        if date in airgo_available_night: report_id.loc[(report_id.date==date).values & (report_id.day_night=='night').values,'airgo'] = True
        if date in bm_available_day: report_id.loc[(report_id.date==date).values & (report_id.day_night=='day').values,'bedmaster'] = True
        if date in bm_available_night: report_id.loc[(report_id.date==date).values & (report_id.day_night=='night').values,'bedmaster'] = True
    
    return report_id


In [4]:
report_id_all_subjects = []

for study_id in list(range(1,190)): #+ list(range(168,189)): # range(1,167):

    study_id = str(study_id).zfill(3)
    bm_available_day, bm_available_night = get_bm_data()
    # print(bm_available_day)
    # print(bm_available_night)
    airgo_available_day, airgo_available_night = get_airgo_data()
    # print(airgo_available_day)
    # print(airgo_available_night)
    nights_in_icu, cam_icu_morning, cam_icu_evening = get_icu_data()
    # print(nights_in_icu)
    # print(cam_icu_morning)
    # print(cam_icu_evening)
    report_id = report_for_id()
    
    print(study_id)
#     print(report_id)
    
    report_id_all_subjects.append(report_id)

001
002
003
004
005
006
007
008
009
010
011
012
013
014
015
016
017
018
019
020
021
022
023
024
025
026
027
028
029
030
031
032
033
034
035
036
037
038
039
040
041
042
043
044
045
046
047
048
049
050
051
052
053
054
055
056
057
058
059
060


C:\Users\wg984\AppData\Local\Continuum\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3254: DtypeWarning:

Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.



061
062
063
064
065
066
067
068
069
070
071
072
073
074
075
076
077
078
079
080
081
082
083
084
085
086
087
088
089
090
091
092
093
094
095
096
097
098
099
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189


In [10]:
report = pd.concat(report_id_all_subjects, ignore_index=True)
report.head()

,study_id,date,day_night,cam_icu,airgo,bedmaster
0,001,2018-06-05,day,False,False,True
1,001,2018-06-05,night,False,False,True
2,001,2018-06-06,day,False,True,True
3,001,2018-06-06,night,False,True,True
4,001,2018-06-07,day,False,False,True


In [11]:
np.unique(report.cam_icu.values)

array([False], dtype=object)

In [7]:
idx_to_drop = report.loc[(report.cam_icu.values==False) & (report.airgo.values==False) & (report.bedmaster.values==False),:].index
report.drop(idx_to_drop, axis=0, inplace=True)
report.to_csv('//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/data_analysis/study/data_availability.csv', index=False)

In [8]:
report = pd.concat(report_id_all_subjects, ignore_index=True)
idx_to_drop = report.loc[report.cam_icu.values==False,:].index
report.drop(idx_to_drop, axis=0, inplace=True)
report.to_csv('//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/data_analysis/study/data_availability_whereCAM.csv', index=False)
report.head()

,study_id,date,day_night,cam_icu,airgo,bedmaster


In [12]:
report = pd.concat(report_id_all_subjects, ignore_index=True)
idx_to_drop = report.loc[np.logical_and(report.cam_icu.values==False, report.airgo.values==False),:].index
report.drop(idx_to_drop, axis=0, inplace=True)
report.to_csv('//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/data_analysis/study/data_availability_whereCAMorAirgo.csv', index=False)

In [13]:
report.head()

,study_id,date,day_night,cam_icu,airgo,bedmaster
2,001,2018-06-06,day,False,True,True
3,001,2018-06-06,night,False,True,True
8,002,2018-06-11,day,False,True,True
9,002,2018-06-11,night,False,True,True
10,002,2018-06-12,day,False,True,True


## do 'report' part, i.e. fill it in one csv

In [277]:
print('nights in the icu:')
print(nights_in_icu)
print('cam icus in the morning:')
print(cam_icu_morning)
print('cam icus in the evening:')
print(cam_icu_evening)
print('bm_available_day')
print(bm_available_day)
print('bm_available_night')
print(bm_available_night)


nights in the icu:
['2018-06-18', '2018-06-19', '2018-06-20']
cam icus in the morning:
['2018-06-19', '2018-06-20', '2018-06-21']
cam icus in the evening:
['2018-06-18', '2018-06-19', '2018-06-20', '2018-06-21']
bm_available_day
['2018-06-18', '2018-06-19', '2018-06-20', '2018-06-21']
bm_available_night
['2018-06-17', '2018-06-18', '2018-06-19', '2018-06-20']
